In [1]:
!nvidia-smi
!pip list 2>/dev/null | grep -i -E "tensorflow|keras|aeon|codecarbon"

Fri Nov 13 21:36:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/MSD-IRIMAS/LITE.git
%cd /content/LITE
from google.colab import userdata
import subprocess
tok = userdata.get("GH_TOKEN")
r = subprocess.run(["git", "clone", f"https://{tok}@github.com/DevRana003/lite-results-65-128.git",
                    "/content/results_store"], capture_output=True, text=True)
print("return code:", r.returncode, r.stderr.replace(tok, "***"))
for k, v in (("user.name", "colab"), ("user.email", "colab@example.com")):
    subprocess.run(["git", "config", k, v], cwd="/content/results_store")

Cloning into 'LITE'...
remote: Enumerating objects: 165, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 165 (delta 30), reused 27 (delta 27), pack-reused 127 (from 1)
Receiving objects: 100% (165/165), 1.92 MiB | 27.73 MiB/s, done.
Resolving deltas: 100% (64/64), done.
/content/LITE
return code: 0 Cloning into '/content/results_store'...



In [3]:
!pip install -q aeon codecarbon setuptools

In [4]:
%cd /content/LITE
!python /content/results_store/patch_lite_es.py
!sed -i 's/n_epochs=2,/n_epochs=1500,/' src/classifiers/lite.py
assert "n_epochs=1500," in open("src/classifiers/lite.py").read(), "epochs patch failed"

/content/LITE
patched; backup in src/classifiers/lite.py.bak


In [5]:
!LITE_EARLY_STOPPING=1 LITE_PATIENCE=100 python main.py --dataset Coffee --runs 1 --track-emissions True --output-directory /content/smoke/
!cat /content/smoke/LITE/run_0/Coffee/dict_emissions.json
!cat /content/smoke/LITE/results_ucr.csv

I0000 00:00:1794606052.560533    1944 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1794606062.414487    1944 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1794606062.416245    1944 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
[codecarbon WARNING @ 21:41:04] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 21:41:04] Emissions data (if any) will be saved to file /content/smoke/LITE/run_0/Coffee/emissions.csv
[codecarbon INFO @ 21:41:04] [setup] RAM Tracking...
[codec

In [6]:
%cd /content/LITE
!sed -i 's/patience=self.patience,/patience=self.patience, min_delta=float(os.environ.get("LITE_MIN_DELTA", "0")),/' src/classifiers/lite.py
!grep -n "min_delta" src/classifiers/lite.py

p = "/content/results_store/run_arms.py"
s = open(p).read()
old = 'LITE_VAL_SPLIT="0.2")'
assert s.count(old) == 1, "runner line not found"
open(p, "w").write(s.replace(old, 'LITE_VAL_SPLIT="0.2", LITE_MIN_DELTA="0.001")'))
print("runner updated")

/content/LITE
290:                patience=self.patience, min_delta=float(os.environ.get("LITE_MIN_DELTA", "0")),
runner updated


In [7]:
!LITE_MIN_DELTA=0.001 LITE_EARLY_STOPPING=1 LITE_PATIENCE=100 python main.py --dataset Coffee --runs 1 --track-emissions True --output-directory /content/smoke2/
!cat /content/smoke2/LITE/run_0/Coffee/dict_emissions.json
!cat /content/smoke2/LITE/results_ucr.csv

I0000 00:00:1794606881.677004   31050 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1794606885.898755   31050 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1794606885.900545   31050 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
[codecarbon WARNING @ 21:54:47] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 21:54:47] Emissions data (if any) will be saved to file /content/smoke2/LITE/run_0/Coffee/emissions.csv
[codecarbon INFO @ 21:54:47] [setup] RAM Tracking...
[code

In [8]:
p = "/content/results_store/run_arms.py"
s = open(p).read()
if '"D_es_p300"' in s:
    print("arm D already present")
else:
    old = 'ARMS = {"A_base": ("0", "100"), "B_split": ("1", "100000"), "C_es": ("1", "100")}'
    assert s.count(old) == 1, "ARMS line not found"
    new = 'ARMS = {"A_base": ("0", "100"), "B_split": ("1", "100000"), "C_es": ("1", "100"), "D_es_p300": ("1", "300")}'
    open(p, "w").write(s.replace(old, new))
    print("arm D added")

arm D added


In [9]:
!grep -n "min_delta" /content/LITE/src/classifiers/lite.py
!grep -n "ARMS =\|MIN_DELTA" /content/results_store/run_arms.py

290:                patience=self.patience, min_delta=float(os.environ.get("LITE_MIN_DELTA", "0")),
8:ARMS = {"A_base": ("0", "100"), "B_split": ("1", "100000"), "C_es": ("1", "100"), "D_es_p300": ("1", "300")}
30:        env = dict(os.environ, LITE_EARLY_STOPPING=es, LITE_PATIENCE=pat, LITE_VAL_SPLIT="0.2", LITE_MIN_DELTA="0.001")


In [10]:
import os
os.environ.update(SESSION_BUDGET_H="4.5", RUNS="1")
%cd /content/LITE
!python /content/results_store/run_arms.py

/content/LITE
A_base ItalyPowerDemand: exit 0, 5.6 min
B_split ItalyPowerDemand: exit 0, 5.0 min
C_es ItalyPowerDemand: exit 0, 2.9 min
D_es_p300 ItalyPowerDemand: exit 0, 3.7 min
A_base SmoothSubspace: exit 0, 4.8 min
B_split SmoothSubspace: exit 0, 5.7 min
C_es SmoothSubspace: exit 0, 4.6 min
D_es_p300 SmoothSubspace: exit 0, 6.2 min
A_base SyntheticControl: exit 0, 6.7 min
B_split SyntheticControl: exit 0, 6.3 min
C_es SyntheticControl: exit 0, 2.3 min
D_es_p300 SyntheticControl: exit 0, 3.3 min
A_base Trace: exit 0, 5.8 min
B_split Trace: exit 0, 5.9 min
C_es Trace: exit 0, 3.6 min
D_es_p300 Trace: exit 0, 4.3 min
A_base SwedishLeaf: exit 0, 7.5 min
B_split SwedishLeaf: exit 0, 6.4 min
C_es SwedishLeaf: exit 0, 3.0 min
D_es_p300 SwedishLeaf: exit 0, 3.5 min
A_base Strawberry: exit 0, 12.1 min
B_split Strawberry: exit 0, 10.6 min
C_es Strawberry: exit 0, 2.4 min
D_es_p300 Strawberry: exit 0, 4.5 min
A_base Symbols: exit 0, 9.4 min
B_split Symbols: exit 0, 6.2 min
C_es Symbols: exit 

In [1]:
import os, subprocess
from google.colab import userdata
if not os.path.isdir("/content/results_store"):
    tok = userdata.get("GH_TOKEN")
    r = subprocess.run(["git", "clone", f"https://{tok}@github.com/DevRana003/lite-results-65-128.git",
                        "/content/results_store"], capture_output=True, text=True)
    print("clone return code:", r.returncode, r.stderr.replace(tok, "***"))
else:
    r = subprocess.run(["git", "pull"], cwd="/content/results_store", capture_output=True, text=True)
    print("pull:", r.stdout[-200:], r.stderr[-200:])
!ls /content/results_store

clone return code: 0 Cloning into '/content/results_store'...

datasets_65_128.txt   README.md        results_D_es_p300  results_pass5
datasets_improve.txt  reorder.py       results_pass1	  run_arms.py
datasets_pass2.txt    results_A_base   results_pass2	  run_shard.py
failed.txt	      results_B_split  results_pass3
patch_lite_es.py      results_C_es     results_pass4


In [2]:
%%writefile /content/analyze4.py
import glob, json, os
import pandas as pd
from scipy.stats import wilcoxon
ST = os.environ.get("ST", "/content/results_store")
ARMS = ["A_base", "B_split", "C_es", "D_es_p300"]
KG = 0.4707   # kg CO2 per kWh, from the Singapore rows (CO2 / energy)
rows = []
for arm in ARMS:
    c = pd.read_csv(f"{ST}/results_{arm}/LITE/results_ucr.csv")
    for _, r in c.iterrows():
        f = glob.glob(f"{ST}/results_{arm}/LITE/run_0/{r['dataset']}/dict_emissions.json")
        j = json.load(open(f[0])) if f else {}
        rows.append(dict(arm=arm, dataset=r["dataset"], acc=r["LITE-mean"], time=r["Training Time-mean"],
                         kwh=r["Energy Consumption-mean"], epochs=j.get("epochs_run"), best=j.get("best_epoch")))
w = pd.DataFrame(rows).pivot(index="dataset", columns="arm")
acc, tm, kw = w["acc"][ARMS], w["time"][ARMS], w["kwh"][ARMS]
print("MEAN ACCURACY\n", acc.mean().round(4).to_string())
print("\nACCURACY CHANGE VS A (per dataset)\n", acc.sub(acc["A_base"], axis=0).round(3).to_string())
for name, tab in (("TRAINING TIME", tm), ("ENERGY", kw)):
    print(f"\n{name} saved, % (vs A, vs B)")
    for a in ("C_es", "D_es_p300"):
        print(" ", a, round(100 * (1 - tab[a].sum() / tab["A_base"].sum()), 1),
              round(100 * (1 - tab[a].sum() / tab["B_split"].sum()), 1))
print("\nCO2 (kg) at constant intensity\n", (kw.sum() * KG).round(4).to_string())
print("\nEPOCHS RUN / BEST EPOCH (C and D)\n", w[["epochs", "best"]].loc[:, (slice(None), ["C_es", "D_es_p300"])].to_string())
for a in ("C_es", "D_es_p300"):
    d = acc[a] - acc["A_base"]
    print(f"\nWilcoxon {a} vs A:", round(wilcoxon(d).pvalue, 3) if (d != 0).any() else "all equal")
w.to_csv("/content/improvement_all.csv")

Writing /content/analyze4.py


In [3]:
!python /content/analyze4.py

MEAN ACCURACY
 arm
A_base       0.9360
B_split      0.9404
C_es         0.8313
D_es_p300    0.9291

ACCURACY CHANGE VS A (per dataset)
 arm                    A_base  B_split   C_es  D_es_p300
dataset                                                 
ItalyPowerDemand          0.0    0.001  0.004     -0.004
ShapeletSim               0.0    0.011 -0.222     -0.033
SmoothSubspace            0.0    0.020  0.007      0.013
SonyAIBORobotSurface1     0.0    0.053  0.032      0.018
SonyAIBORobotSurface2     0.0   -0.006 -0.016      0.000
Strawberry                0.0   -0.027 -0.027     -0.019
SwedishLeaf               0.0   -0.010 -0.016     -0.030
Symbols                   0.0    0.002 -0.808     -0.014
SyntheticControl          0.0    0.000  0.000      0.000
Trace                     0.0    0.000  0.000      0.000

TRAINING TIME saved, % (vs A, vs B)
  C_es 65.7 57.3
  D_es_p300 43.5 29.8

ENERGY saved, % (vs A, vs B)
  C_es 67.3 58.8
  D_es_p300 46.1 32.1

CO2 (kg) at constant intensity
 ar

In [4]:
import subprocess, pandas as pd
from scipy.stats import wilcoxon
subprocess.run(["git", "clone", "-q", "https://github.com/MSD-IRIMAS/LITE.git", "/content/LITE_src"])
paper = pd.read_csv("/content/LITE_src/results.csv")[["dataset", "LITE"]]
frames = []
for i in range(1, 6):
    d = pd.read_csv(f"/content/results_store/results_pass{i}/LITE/results_ucr.csv")[["dataset", "LITE-mean"]]
    frames.append(d)
g = pd.concat(frames).groupby("dataset")["LITE-mean"]
df = pd.DataFrame({"mine": g.mean(), "n": g.count(), "lo": g.min(), "hi": g.max()}).reset_index().merge(paper, on="dataset")
df["diff"] = df["mine"] - df["LITE"]; df["spread"] = df["hi"] - df["lo"]
print("datasets:", len(df), "| runs per dataset:", df["n"].value_counts().to_dict())
print(f"mean accuracy: mine {df['mine'].mean():.4f}  paper {df['LITE'].mean():.4f}  diff {df['diff'].mean():+.4f}  mean|diff| {df['diff'].abs().mean():.4f}")
print(f"mine higher on {(df['diff'] > 0).sum()}, lower on {(df['diff'] < 0).sum()}; Wilcoxon p = {wilcoxon(df['diff']).pvalue:.4f}")
print(f"mean run-to-run spread (max-min): {df['spread'].mean():.4f}")
print(f"paper mean outside my [min,max] range on {((df['LITE'] < df['lo']) | (df['LITE'] > df['hi'])).sum()} of {len(df)} datasets")
print(df.reindex(df['diff'].abs().sort_values(ascending=False).index).head(10).round(4).to_string(index=False))

datasets: 60 | runs per dataset: {5: 59, 4: 1}
mean accuracy: mine 0.8339  paper 0.8334  diff +0.0005  mean|diff| 0.0075
mine higher on 28, lower on 30; Wilcoxon p = 0.7131
mean run-to-run spread (max-min): 0.0391
paper mean outside my [min,max] range on 8 of 60 datasets
                     dataset   mine  n     lo     hi   LITE    diff  spread
                    OliveOil 0.7400  5 0.5000 0.9000 0.6933  0.0467  0.4000
                        Wine 0.5963  5 0.5556 0.6296 0.6407 -0.0444  0.0741
         SemgHandMovementCh2 0.4418  5 0.4222 0.4511 0.4787 -0.0369  0.0289
                 ShapeletSim 0.7111  5 0.6833 0.7333 0.7378 -0.0267  0.0500
                  Lightning2 0.7787  4 0.7213 0.8361 0.7541  0.0246  0.1148
                   PowerCons 0.9667  5 0.9444 0.9778 0.9444  0.0222  0.0333
       SonyAIBORobotSurface1 0.8636  5 0.8236 0.8852 0.8416  0.0220  0.0616
        RefrigerationDevices 0.5125  5 0.4747 0.5307 0.4965  0.0160  0.0560
                  ScreenType 0.5797  5 0.546

In [5]:
import pandas as pd
paper = pd.read_csv("/content/LITE_src/results.csv")
mine = pd.concat([pd.read_csv(f"/content/results_store/results_pass{i}/LITE/results_ucr.csv")[["dataset", "LITE-mean"]]
                  for i in range(1, 6)]).groupby("dataset")["LITE-mean"].mean().rename("mine")
t = paper.merge(mine, on="dataset")
print(t[t["dataset"].isin(["OliveOil", "ShapeletSim", "PigAirwayPressure"])]
      [["dataset", "mine", "LITE", "Inception", "InceptionTime", "ROCKET"]].round(3).to_string(index=False))
todo = [x.strip() for x in open("/content/results_store/datasets_65_128.txt") if x.strip()]
print("not in my results:", [x for x in todo if x not in set(mine.index)])

          dataset  mine  LITE  Inception  InceptionTime  ROCKET
         OliveOil 0.740 0.693      0.847          0.867   0.917
PigAirwayPressure 0.356 0.347      0.513          0.543   0.095
      ShapeletSim 0.711 0.738      0.952          0.989   1.000
not in my results: ['PLAID', 'PickupGestureWiimoteZ', 'ShakeGestureWiimoteZ', 'MelbournePedestrian']
